# 02 · Data Cleaning & Quality
Runs the reusable cleaning pipeline (`src/data_cleaning`) and reviews every decision it logged.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
from IPython.display import Image, Markdown, display
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200); pd.set_option("display.float_format", "{:,.2f}".format)
from src.data_cleaning.loader import load_all_processed, load_all_raw
from src.kpi.kpis import kpi_by, compute_all_kpis
from src.analytics import eda
d = load_all_processed()
t, f, o, c = d["tickets_enriched"], d["customer_feedback"], d["operations"], d["customers"]
VIS = ROOT / "visualizations"
def show(*names):
    for n in names: display(Image(filename=str(VIS / f"{n}.png"), width=820))

In [2]:
from src.data_cleaning.pipeline import run_cleaning, quality_scores
raw = load_all_raw()
clean, log = run_cleaning(raw)
issues = log.to_frame()
issues[issues.records_affected > 0]

,table,check,issue_type,records_affected,action,decision
0,customers,exact_duplicates,Duplicate,30,removed,Rows identical in every column are copies of t...
1,customers,segment_label_case,Inconsistent label,40,corrected,Segment labels standardised to the four canoni...
2,customers,registration_date_format,Data type / format,60,corrected,Legacy 'DD-Mon-YYYY' dates converted to ISO da...
3,customers,missing_region,Missing value,45,corrected,"Region is fully determined by country, so miss..."
4,customers,missing_clv,Missing value,70,imputed,CLV imputed with the segment median (CLV diffe...
5,support_tickets,exact_duplicates,Duplicate,150,removed,Rows identical in every column are copies of t...
6,support_tickets,channel_label_variants,Inconsistent label,400,corrected,"Channel spellings (e.g. 'E-mail', 'chat') mapp..."
7,support_tickets,missing_channel,Missing value,120,imputed,Channel cannot be inferred from other fields; ...
8,support_tickets,priority_label_case,Inconsistent label,300,corrected,Priority labels standardised to Title Case.
9,support_tickets,sla_target_vs_priority,Inconsistent value,180,corrected,SLA target is a business rule of priority (Cri...


## Quality score before vs after

In [3]:
scores = quality_scores(raw, clean, log)
scores[["table", "raw_records", "clean_records", "records_removed", "records_corrected", "records_imputed", "records_flagged", "quality_score_before", "quality_score_after"]]

,table,raw_records,clean_records,records_removed,records_corrected,records_imputed,records_flagged,quality_score_before,quality_score_after
0,customers,3230,3200,30,145,70,0,98.36,100.00
1,departments,6,6,0,0,0,0,100.00,100.00
2,agents,60,60,0,0,0,0,100.00,100.00
3,support_tickets,24150,24000,150,880,120,130,98.47,99.98
4,customer_feedback,12538,12466,72,25,0,150,99.48,99.93
5,operations,14248,14248,0,32,88,0,99.87,100.00
6,agent_performance,1984,1984,0,0,0,8,99.88,100.00
7,business_costs,1296,1296,0,40,0,0,99.34,100.00
8,calendar,1826,1826,0,0,0,0,100.00,100.00


## Verification: did cleaning catch every injected defect?

In [4]:
m = json.loads((ROOT / "data/raw/_generation_manifest.json").read_text()) if "json" in dir() else __import__("json").loads((ROOT / "data/raw/_generation_manifest.json").read_text())
lk = issues.set_index(["table", "check"]).records_affected
checks = {
 "tickets_exact_duplicates": ("support_tickets", "exact_duplicates"),
 "tickets_resolved_before_created": ("support_tickets", "resolved_before_created"),
 "tickets_orphan_agent": ("support_tickets", "fk_agent"),
 "tickets_wrong_sla_target": ("support_tickets", "sla_target_vs_priority"),
 "feedback_out_of_range_score": ("customer_feedback", "satisfaction_scale"),
 "feedback_orphan_ticket": ("customer_feedback", "fk_ticket"),
 "operations_cost_outliers": ("operations", "cost_outlier_robust_z"),
 "operations_negative_cost": ("operations", "negative_cost"),
 "customers_missing_region": ("customers", "missing_region"),
}
pd.DataFrame([{"defect": k, "injected": m[k], "detected": int(lk[v])} for k, v in checks.items()])

,defect,injected,detected
0,tickets_exact_duplicates,150,150
1,tickets_resolved_before_created,25,25
2,tickets_orphan_agent,15,15
3,tickets_wrong_sla_target,180,180
4,feedback_out_of_range_score,40,40
5,feedback_orphan_ticket,12,12
6,operations_cost_outliers,18,18
7,operations_negative_cost,12,12
8,customers_missing_region,45,45


## Key cleaning decisions
| Issue | Decision | Why |
|---|---|---|
| Exact duplicates | removed | identical rows are copies of the same event |
| Resolved before created | timestamp nulled, ticket kept, flagged | keeps volume KPIs right; excluded from time/SLA KPIs |
| Orphan agent IDs | set to NULL, flagged | ticket still counts; agent KPIs exclude it |
| Wrong SLA target | recomputed from priority | SLA target is a business rule of priority |
| CSAT outside 1-5 | nulled, flagged | cannot be mapped; never imputed |
| Cost outliers (robust z > 8) | replaced by process-month median cost/txn × volume, original kept | 10x+ normal values are entry errors |
| Missing region | derived from country | exact derivation, not an estimate |
| Missing CLV | segment median + `clv_imputed` flag | segments differ ~60x in CLV |

Every injected defect was detected with the exact injected count (table above).